# LeetCode 3197 — Find the Minimum Area to Cover All Ones II bằng Python

- Link: https://leetcode.com/problems/find-the-minimum-area-to-cover-all-ones-ii/
- Độ khó: **Hard**
- Chủ đề: **Liệt kê hình dạng phân hoạch** — ba hình chữ nhật không chồng nhau luôn tách được bằng nhát cắt thẳng ("guillotine") ⇒ chỉ có **6 kiểu** chia lưới (3 dải ngang, 3 dải dọc, 4 kiểu chữ T); mỗi phần lấy **hình chữ nhật bao** các số 1

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def minimumSum(self, grid: List[List[int]]) -> int:
```

## 1. Mục tiêu học

1. Bài con: hình chữ nhật **nhỏ nhất** phủ mọi số 1 trong một vùng là **hình bao** (bounding box) `(max hàng − min hàng + 1) × (max cột − min cột + 1)` (LC 3195).
2. Ba hình chữ nhật rời nhau (trục song song) luôn có một đường thẳng ngang/dọc tách một hình khỏi hai hình còn lại ⇒ cấu hình tối ưu ứng với một trong **6 kiểu** chia lưới.
3. Duyệt mọi vị trí cắt: `O(n² + m² + n·m)` cách chia, mỗi cách tính 3 hình bao.
4. Tổng quát: DP guillotine `f(vùng, k)` cho `k` hình (đúng với `k ≤ 4`).
5. Kiểm chứng bằng brute liệt kê bộ ba hình chữ nhật trên lưới nhỏ.

## 2. Đề bài & ví dụ

Cho lưới nhị phân `grid`. Tìm **3** hình chữ nhật (cạnh song song trục, diện tích khác 0, **không chồng nhau** — được chạm cạnh) sao cho mọi ô `1` nằm trong một hình nào đó. Trả về tổng diện tích nhỏ nhất.

**Ví dụ 1:** `grid = [[1,0,1],[1,1,1]]` → `5` (`2 + 2 + 1`).

**Ví dụ 2:** `grid = [[1,0,1,0],[0,1,0,1]]` → `5` (`3 + 1 + 1`).

**Ràng buộc:** `1 ≤ n, m ≤ 30`; `grid[i][j] ∈ {0, 1}`; có ít nhất ba số `1`.

In [1]:
from typing import List
from functools import lru_cache
import itertools, random, time

examples = [([[1, 0, 1], [1, 1, 1]], 5), ([[1, 0, 1, 0], [0, 1, 0, 1]], 5)]
for g, exp in examples:
    for row in g:
        print(' '.join(map(str, row)))
    print('-> kỳ vọng', exp); print()

1 0 1
1 1 1
-> kỳ vọng 5

1 0 1 0
0 1 0 1
-> kỳ vọng 5



## 3. Kiến thức nền

### 3.1. Hình bao của các số 1 trong một vùng

Vùng `[r1..r2] × [c1..c2]`. Biểu diễn mỗi hàng bằng **bitmask** các cột có số 1; AND với mặt nạ cột `[c1..c2]` cho biết hàng có số 1 trong vùng không; OR các hàng lại cho tập cột ⇒ cột trái/phải bằng thao tác bit. Mỗi lần gọi `O(số hàng)`.

Quy ước: vùng **không có** số 1 trả về `0` (xem mục 10 vì sao an toàn).

In [2]:
def make_area(grid):
    rows = [sum(1 << j for j, x in enumerate(r) if x) for r in grid]
    def area(r1, r2, c1, c2):
        if r1 > r2 or c1 > c2:
            return 0
        msk = ((1 << (c2 + 1)) - 1) ^ ((1 << c1) - 1)
        top = -1; bot = 0; acc = 0
        for r in range(r1, r2 + 1):
            x = rows[r] & msk
            if x:
                if top < 0:
                    top = r
                bot = r; acc |= x
        if not acc:
            return 0
        left = (acc & -acc).bit_length() - 1; right = acc.bit_length() - 1
        return (bot - top + 1) * (right - left + 1)
    return area

A = make_area(examples[0][0])
print('hình bao toàn lưới ví dụ 1:', A(0, 1, 0, 2), '| cột 0:', A(0, 1, 0, 0), '| cột 1:', A(0, 1, 1, 1))

hình bao toàn lưới ví dụ 1: 6 | cột 0: 2 | cột 1: 1


### 3.2. Vì sao chỉ có 6 kiểu chia

Với ba hình chữ nhật rời nhau, luôn tồn tại một đường thẳng ngang hoặc dọc tách được **một** hình khỏi hai hình kia (với ≤ 4 hình luôn có nhát cắt guillotine; phản ví dụ "chong chóng" cần 5 hình). Hai hình còn lại lại tách được bằng một nhát cắt nữa. Kết hợp hướng của hai nhát:

1. Ba dải ngang. 2. Ba dải dọc.
3. Dải trên + nửa dưới chia dọc. 4. Dải dưới + nửa trên chia dọc.
5. Dải trái + nửa phải chia ngang. 6. Dải phải + nửa trái chia ngang.

(Hình bao của các số 1 trong mỗi phần nằm gọn trong phần đó nên các hình không chồng nhau.)

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Thay vì tìm 3 hình chữ nhật (không gian khổng lồ), ta tìm **2 nhát cắt** chia lưới thành 3 vùng; mỗi vùng tự lấy hình bao của các số 1 bên trong. Ví dụ 1: chia 3 dải dọc → cột 0 (bao `2`), cột 1 (bao `1`), cột 2 (bao `2`) ⇒ `5`.

## 5. Cách 1 — Brute Force: thử mọi bộ ba hình chữ nhật

In [3]:
def brute(grid):
    n, m = len(grid), len(grid[0])
    R = [(r1, r2, c1, c2) for r1 in range(n) for r2 in range(r1, n) for c1 in range(m) for c2 in range(c1, m)]
    C = [frozenset((i, j) for i in range(r[0], r[1] + 1) for j in range(r[2], r[3] + 1)) for r in R]
    ones = [(i, j) for i in range(n) for j in range(m) if grid[i][j]]
    best = None
    for a, b, c in itertools.combinations(range(len(R)), 3):
        if C[a] & C[b] or C[a] & C[c] or C[b] & C[c]:
            continue
        U = C[a] | C[b] | C[c]
        if all(o in U for o in ones):
            s = len(C[a]) + len(C[b]) + len(C[c])
            best = s if best is None else min(best, s)
    return best

for g, exp in examples:
    print('brute:', brute(g), '| kỳ vọng', exp)

brute: 5 | kỳ vọng 5
brute: 5 | kỳ vọng 5


**Độ phức tạp:** `O(R³ · nm)` với `R = O(n²m²)` hình chữ nhật — chỉ cho lưới `≤ 3 × 4`. Brute **không** giả sử gì về hình dạng, nên kiểm chứng được cả lập luận 6 kiểu.

## 6. Ý tưởng thuật toán / bước đột phá

```
best = ∞
for i < j (hàng):  ba dải ngang
for i < j (cột):   ba dải dọc
for i (hàng), j (cột): 4 kiểu chữ T theo hàng i / cột j
mỗi phần: area(vùng) = hình bao các số 1
```

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Liệt kê trực tiếp 6 kiểu (lời giải chính)

In [4]:
def six_shapes(grid):
    n, m = len(grid), len(grid[0]); A = make_area(grid); best = float('inf')
    for i in range(n - 2):                                   # 1. ba dải ngang
        for j in range(i + 1, n - 1):
            best = min(best, A(0, i, 0, m - 1) + A(i + 1, j, 0, m - 1) + A(j + 1, n - 1, 0, m - 1))
    for i in range(m - 2):                                   # 2. ba dải dọc
        for j in range(i + 1, m - 1):
            best = min(best, A(0, n - 1, 0, i) + A(0, n - 1, i + 1, j) + A(0, n - 1, j + 1, m - 1))
    for i in range(n - 1):
        for j in range(m - 1):
            best = min(best, A(0, i, 0, m - 1) + A(i + 1, n - 1, 0, j) + A(i + 1, n - 1, j + 1, m - 1))  # 3
            best = min(best, A(i + 1, n - 1, 0, m - 1) + A(0, i, 0, j) + A(0, i, j + 1, m - 1))          # 4
            best = min(best, A(0, n - 1, 0, j) + A(0, i, j + 1, m - 1) + A(i + 1, n - 1, j + 1, m - 1))  # 5
            best = min(best, A(0, n - 1, j + 1, m - 1) + A(0, i, 0, j) + A(i + 1, n - 1, 0, j))          # 6
    return best

for g, exp in examples:
    print('6 kiểu:', six_shapes(g), '| kỳ vọng', exp)

6 kiểu: 5 | kỳ vọng 5
6 kiểu: 5 | kỳ vọng 5


### 7.2. Cách 3 — Xoay lưới: chỉ viết 2 kiểu, xoay 4 lần

Kiểu "ba dải ngang" và "dải trên + đáy chia dọc"; xoay lưới 90° bốn lần sinh ra đủ 6 kiểu (có trùng, không sao).

In [5]:
def rotate(g):
    return [list(r) for r in zip(*g[::-1])]

def two_shapes(grid):
    n, m = len(grid), len(grid[0]); A = make_area(grid); best = float('inf')
    for i in range(n - 2):
        for j in range(i + 1, n - 1):
            best = min(best, A(0, i, 0, m - 1) + A(i + 1, j, 0, m - 1) + A(j + 1, n - 1, 0, m - 1))
    for i in range(n - 1):
        for j in range(m - 1):
            best = min(best, A(0, i, 0, m - 1) + A(i + 1, n - 1, 0, j) + A(i + 1, n - 1, j + 1, m - 1))
    return best

def by_rotation(grid):
    best = float('inf'); g = grid
    for _ in range(4):
        best = min(best, two_shapes(g)); g = rotate(g)
    return best

for g, exp in examples:
    print('xoay:', by_rotation(g), '| kỳ vọng', exp)

xoay: 5 | kỳ vọng 5
xoay: 5 | kỳ vọng 5


### 7.3. Cách 4 — DP guillotine tổng quát `f(vùng, k)`

`f(vùng, 1) = area(vùng)`; `f(vùng, k) = min` qua mọi nhát cắt ngang/dọc và mọi cách chia `k = t + (k − t)`. Với `k = 3` chính là 6 kiểu; với `k ≤ 4` vẫn đúng; với `k ≥ 5` có thể sai (cấu hình chong chóng).

In [6]:
def guillotine(grid, K=3):
    n, m = len(grid), len(grid[0]); A = make_area(grid)
    @lru_cache(maxsize=None)
    def f(r1, r2, c1, c2, k):
        if k == 1:
            return A(r1, r2, c1, c2)
        best = float('inf')
        for i in range(r1, r2):
            for t in range(1, k):
                best = min(best, f(r1, i, c1, c2, t) + f(i + 1, r2, c1, c2, k - t))
        for j in range(c1, c2):
            for t in range(1, k):
                best = min(best, f(r1, r2, c1, j, t) + f(r1, r2, j + 1, c2, k - t))
        return best
    return f(0, n - 1, 0, m - 1, K)

for g, exp in examples:
    print('guillotine:', guillotine(g), '| kỳ vọng', exp)

guillotine: 5 | kỳ vọng 5
guillotine: 5 | kỳ vọng 5


Số trạng thái `O(n² m² k)`, mỗi trạng thái `O((n + m) k)` ⇒ với lưới `30 × 30` khoảng `10⁸` — quá chậm cho Python ở cỡ lớn, nhưng là công cụ đối chiếu tốt cho lưới vừa.

## 8. TRACE từng bước

In kiểu chia tốt nhất (kiểu, vị trí cắt, diện tích từng phần) cho hai ví dụ và một lưới chữ T.

In [7]:
def best_with_shape(grid):
    n, m = len(grid), len(grid[0]); A = make_area(grid); cands = []
    for i in range(n - 2):
        for j in range(i + 1, n - 1):
            p = (A(0, i, 0, m - 1), A(i + 1, j, 0, m - 1), A(j + 1, n - 1, 0, m - 1)); cands.append((sum(p), f'3 dải ngang cắt sau hàng {i}, {j}', p))
    for i in range(m - 2):
        for j in range(i + 1, m - 1):
            p = (A(0, n - 1, 0, i), A(0, n - 1, i + 1, j), A(0, n - 1, j + 1, m - 1)); cands.append((sum(p), f'3 dải dọc cắt sau cột {i}, {j}', p))
    for i in range(n - 1):
        for j in range(m - 1):
            p = (A(0, i, 0, m - 1), A(i + 1, n - 1, 0, j), A(i + 1, n - 1, j + 1, m - 1)); cands.append((sum(p), f'T: dải trên tới hàng {i}, đáy chia sau cột {j}', p))
            p = (A(i + 1, n - 1, 0, m - 1), A(0, i, 0, j), A(0, i, j + 1, m - 1)); cands.append((sum(p), f'T: dải dưới từ hàng {i + 1}, đỉnh chia sau cột {j}', p))
            p = (A(0, n - 1, 0, j), A(0, i, j + 1, m - 1), A(i + 1, n - 1, j + 1, m - 1)); cands.append((sum(p), f'T: dải trái tới cột {j}, phải chia sau hàng {i}', p))
            p = (A(0, n - 1, j + 1, m - 1), A(0, i, 0, j), A(i + 1, n - 1, 0, j)); cands.append((sum(p), f'T: dải phải từ cột {j + 1}, trái chia sau hàng {i}', p))
    return min(cands)

g3 = [[1, 0, 1], [0, 0, 0], [1, 1, 1]]
for g in [examples[0][0], examples[1][0], g3]:
    print(g, '->', best_with_shape(g))

[[1, 0, 1], [1, 1, 1]] -> (5, '3 dải dọc cắt sau cột 0, 1', (2, 1, 2))
[[1, 0, 1, 0], [0, 1, 0, 1]] -> (5, 'T: dải dưới từ hàng 1, đỉnh chia sau cột 0', (3, 1, 1))
[[1, 0, 1], [0, 0, 0], [1, 1, 1]] -> (5, 'T: dải dưới từ hàng 1, đỉnh chia sau cột 0', (3, 1, 1))


## 9. Code nộp LeetCode

In [8]:
class Solution:
    def minimumSum(self, grid: List[List[int]]) -> int:
        n, m = len(grid), len(grid[0])
        rows = [sum(1 << j for j, x in enumerate(r) if x) for r in grid]

        def A(r1, r2, c1, c2):
            msk = ((1 << (c2 + 1)) - 1) ^ ((1 << c1) - 1)
            top = -1
            bot = 0
            acc = 0
            for r in range(r1, r2 + 1):
                x = rows[r] & msk
                if x:
                    if top < 0:
                        top = r
                    bot = r
                    acc |= x
            if not acc:
                return 0
            return (bot - top + 1) * (acc.bit_length() - (acc & -acc).bit_length() + 1)

        best = float('inf')
        for i in range(n - 2):
            for j in range(i + 1, n - 1):
                best = min(best, A(0, i, 0, m - 1) + A(i + 1, j, 0, m - 1) + A(j + 1, n - 1, 0, m - 1))
        for i in range(m - 2):
            for j in range(i + 1, m - 1):
                best = min(best, A(0, n - 1, 0, i) + A(0, n - 1, i + 1, j) + A(0, n - 1, j + 1, m - 1))
        for i in range(n - 1):
            for j in range(m - 1):
                best = min(best,
                           A(0, i, 0, m - 1) + A(i + 1, n - 1, 0, j) + A(i + 1, n - 1, j + 1, m - 1),
                           A(i + 1, n - 1, 0, m - 1) + A(0, i, 0, j) + A(0, i, j + 1, m - 1),
                           A(0, n - 1, 0, j) + A(0, i, j + 1, m - 1) + A(i + 1, n - 1, j + 1, m - 1),
                           A(0, n - 1, j + 1, m - 1) + A(0, i, 0, j) + A(i + 1, n - 1, 0, j))
        return best

for g, exp in examples:
    got = Solution().minimumSum(g)
    print(got, 'OK' if got == exp else 'SAI')

5 OK
5 OK


## 10. Vì sao thuật toán đúng

1. **Mọi kiểu chia cho lời giải hợp lệ (cận trên):** ba hình bao nằm trong ba vùng rời nhau ⇒ không chồng nhau, phủ hết số 1. Nếu một vùng rỗng (bao = 0), ta vẫn có hai hình phủ hết; vì có `≥ 3` số 1, một hình chứa `≥ 2` số 1 có thể cắt đôi theo một đường giữa hai số 1 thành hai hình bao có tổng diện tích **không lớn hơn** ⇒ luôn có lời giải 3 hình với tổng `≤` giá trị đó. Vậy dùng `0` cho vùng rỗng không làm đáp án nhỏ hơn thực tế.
2. **Lời giải tối ưu thuộc một kiểu (cận dưới):** ba hình rời nhau luôn tách được bằng một đường thẳng (guillotine với ≤ 4 hình), rồi hai hình còn lại tách bằng đường thứ hai ⇒ cấu hình nằm trong một kiểu chia; thay mỗi hình bằng hình bao số 1 trong vùng của nó chỉ làm diện tích giảm.
3. Brute ở mục 12 (không giả thiết hình dạng) khớp với 6 kiểu trên mọi lưới nhỏ thử.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Brute bộ ba | `O(n⁶m⁶ · nm)` | `O(n²m² · nm)` |
| 2. 6 kiểu | `O((n² + m² + nm) · n)` | `O(n)` |
| 3. Xoay 4 lần | như cách 2 (×2) | `O(nm)` |
| 4. DP guillotine | `O(n²m² (n+m) k²)` | `O(n²m²k)` |

Với `30 × 30`: cách 2 khoảng `(900 + 900 + 4·900) · 30 ≈ 1.6·10⁵` bước bit — rất nhanh.

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [9]:
impls = {'six': six_shapes, 'rotation': by_rotation, 'guillotine': guillotine,
         'submit': lambda g: Solution().minimumSum(g)}
edge = [([[1, 1, 1]], 3), ([[1], [1], [1]], 3), ([[1, 1], [1, 0]], 3), ([[1, 1], [1, 1]], 4),
        ([[1, 0, 1], [0, 0, 0], [1, 1, 1]], 5), ([[1, 0, 0], [0, 1, 0], [0, 0, 1]], 3)]
for g, exp in examples + edge:
    assert brute(g) == exp, (g, brute(g))
    for name, f in impls.items():
        assert f(g) == exp, (name, g)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với brute (lưới ≤ `3 × 4`)

In [10]:
rng = random.Random(3197)
cnt = 0
for _ in range(220):
    n = rng.randint(1, 3); m = rng.randint(1, 4)
    if n * m < 3:
        continue
    g = [[int(rng.random() < 0.5) for _ in range(m)] for _ in range(n)]
    if sum(map(sum, g)) < 3:
        continue
    exp = brute(g)
    for name, f in impls.items():
        assert f(g) == exp, (name, g)
    cnt += 1
print('Khớp brute trên', cnt, 'lưới ngẫu nhiên.')

Khớp brute trên 96 lưới ngẫu nhiên.


In [11]:
# Lưới vừa: so 6 kiểu với DP guillotine (không cần brute)
for _ in range(25):
    n = rng.randint(2, 8); m = rng.randint(2, 8)
    g = [[int(rng.random() < 0.3) for _ in range(m)] for _ in range(n)]
    if sum(map(sum, g)) < 3:
        continue
    assert Solution().minimumSum(g) == guillotine(g) == by_rotation(g)
print('6 kiểu = DP guillotine = xoay trên lưới đến 8 × 8.')

6 kiểu = DP guillotine = xoay trên lưới đến 8 × 8.


### 12.3. Ca lớn `30 × 30`

In [12]:
big = {
    'ngẫu nhiên 50%': [[rng.randint(0, 1) for _ in range(30)] for _ in range(30)],
    'toàn 1': [[1] * 30 for _ in range(30)],
    'ba điểm góc': [[1 if (i, j) in {(0, 0), (0, 29), (29, 15)} else 0 for j in range(30)] for i in range(30)],
}
for name, g in big.items():
    t = time.time(); x = Solution().minimumSum(g); t1 = time.time() - t
    y = by_rotation(g)
    assert x == y
    print(f'{name:16s}: {x:4d} | nộp {t1:.3f}s')
assert Solution().minimumSum(big['toàn 1']) == 900 and Solution().minimumSum(big['ba điểm góc']) == 3
print('Toàn 1 -> 900; ba điểm rời -> 3: đúng.')

ngẫu nhiên 50%  :  890 | nộp 0.021s
toàn 1          :  900 | nộp 0.021s
ba điểm góc     :    3 | nộp 0.012s
Toàn 1 -> 900; ba điểm rời -> 3: đúng.


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ xét ba dải song song (quên kiểu chữ T)

In [13]:
def wrong_strips_only(grid):
    n, m = len(grid), len(grid[0]); A = make_area(grid); best = float('inf')
    for i in range(n - 2):
        for j in range(i + 1, n - 1):
            best = min(best, A(0, i, 0, m - 1) + A(i + 1, j, 0, m - 1) + A(j + 1, n - 1, 0, m - 1))
    for i in range(m - 2):
        for j in range(i + 1, m - 1):
            best = min(best, A(0, n - 1, 0, i) + A(0, n - 1, i + 1, j) + A(0, n - 1, j + 1, m - 1))
    return best                                                       # SAI

g = [[1, 0, 1], [0, 0, 0], [1, 1, 1]]
print('sai :', wrong_strips_only(g), '| đúng:', Solution().minimumSum(g))

sai : 6 | đúng: 5


**Vì sao sai:** tối ưu là hai ô đơn ở hàng 0 (chia dọc) và dải hàng 2 — kiểu chữ T, không phải ba dải song song.

### Lỗi 2 — Chỉ xét kiểu chữ T (quên ba dải song song)

In [14]:
def wrong_T_only(grid):
    n, m = len(grid), len(grid[0]); A = make_area(grid); best = float('inf')
    for i in range(n - 1):
        for j in range(m - 1):
            best = min(best,
                       A(0, i, 0, m - 1) + A(i + 1, n - 1, 0, j) + A(i + 1, n - 1, j + 1, m - 1),
                       A(i + 1, n - 1, 0, m - 1) + A(0, i, 0, j) + A(0, i, j + 1, m - 1),
                       A(0, n - 1, 0, j) + A(0, i, j + 1, m - 1) + A(i + 1, n - 1, j + 1, m - 1),
                       A(0, n - 1, j + 1, m - 1) + A(0, i, 0, j) + A(i + 1, n - 1, 0, j))
    return best                                                       # SAI

g = [[1, 1, 1], [0, 0, 0], [1, 1, 1], [0, 0, 0], [1, 1, 1]]
print('sai :', wrong_T_only(g), '| đúng:', Solution().minimumSum(g))
print('lưới một hàng:', wrong_T_only([[1, 1, 1]]), '| đúng:', Solution().minimumSum([[1, 1, 1]]))

sai : 12 | đúng: 9
lưới một hàng: inf | đúng: 3


**Vì sao sai:** ba hàng số 1 cách nhau cần đúng ba dải ngang (`3 + 3 + 3`); với lưới một hàng, kiểu chữ T thậm chí không tồn tại (trả về vô cực).

### Lỗi 3 — Gom số 1 thành 3 nhóm tuỳ ý rồi cộng hình bao (quên điều kiện không chồng nhau)

In [15]:
def wrong_allow_overlap(grid):
    ones = [(i, j) for i in range(len(grid)) for j in range(len(grid[0])) if grid[i][j]]
    best = float('inf')
    for lab in itertools.product(range(3), repeat=len(ones)):
        if len(set(lab)) < 3:
            continue
        t = 0
        for c in range(3):
            pts = [o for o, l in zip(ones, lab) if l == c]
            t += (max(p[0] for p in pts) - min(p[0] for p in pts) + 1) * (max(p[1] for p in pts) - min(p[1] for p in pts) + 1)
        best = min(best, t)
    return best                                                       # SAI

g = [[0] * 6 for _ in range(5)]
for k in range(5):
    g[2][k] = 1; g[k][2] = 1
g[0][5] = 1
print('sai :', wrong_allow_overlap(g), '| đúng:', Solution().minimumSum(g))

sai : 11 | đúng: 15


**Vì sao sai:** hình dấu cộng: thanh ngang (5) + thanh dọc bỏ tâm (vẫn có hình bao dài 5, **cắt chéo** thanh ngang) + ô lẻ (1) = `11` — nhưng hai thanh chồng nhau ở tâm, không hợp lệ. Đáp án đúng `15`.

### Lỗi 4 — Sai biên vòng lặp cắt (bỏ vị trí cắt ngay sau hàng/cột 0, ví dụ `range(1, n − 2)`)

In [16]:
def wrong_off_by_one(grid):
    n, m = len(grid), len(grid[0]); A = make_area(grid); best = float('inf')
    for i in range(1, n - 2):                                          # SAI: bỏ i = 0
        for j in range(i + 1, n - 1):
            best = min(best, A(0, i, 0, m - 1) + A(i + 1, j, 0, m - 1) + A(j + 1, n - 1, 0, m - 1))
    for i in range(1, m - 2):                                          # SAI: bỏ i = 0
        for j in range(i + 1, m - 1):
            best = min(best, A(0, n - 1, 0, i) + A(0, n - 1, i + 1, j) + A(0, n - 1, j + 1, m - 1))
    for i in range(n - 1):
        for j in range(m - 1):
            best = min(best,
                       A(0, i, 0, m - 1) + A(i + 1, n - 1, 0, j) + A(i + 1, n - 1, j + 1, m - 1),
                       A(i + 1, n - 1, 0, m - 1) + A(0, i, 0, j) + A(0, i, j + 1, m - 1),
                       A(0, n - 1, 0, j) + A(0, i, j + 1, m - 1) + A(i + 1, n - 1, j + 1, m - 1),
                       A(0, n - 1, j + 1, m - 1) + A(0, i, 0, j) + A(i + 1, n - 1, 0, j))
    return best

print('sai :', wrong_off_by_one([[1, 1, 1]]), '| đúng:', Solution().minimumSum([[1, 1, 1]]))

sai : inf | đúng: 3


**Vì sao sai:** nhát cắt đầu tiên có thể ngay sau hàng/cột 0. Lỗi biên kiểu này thường chỉ lộ ra ở lưới mỏng.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Chia thành k hình chữ nhật không chồng nhau" với `k` nhỏ ⇒ liệt kê **kiểu cắt guillotine**.
- Hình chữ nhật nhỏ nhất phủ tập điểm = hình bao (LC 3195).
- Giảm số trường hợp phải viết bằng **xoay lưới**.
- Lưới `≤ 30 × 30`: bitmask hàng giúp tính hình bao nhanh.

## 15. Biến thể của bài

1. **LC 3195 (bản I):** một hình chữ nhật ⇒ hình bao.
2. Hai hình chữ nhật ⇒ một nhát cắt (bài tập 4).
3. `k` hình với `k ≤ 4` ⇒ DP guillotine; `k ≥ 5` cần xét cả cấu hình chong chóng.
4. Tối thiểu **chu vi** thay vì diện tích ⇒ cùng khung, đổi hàm `area`.

In [17]:
# Biến thể 4: tổng chu vi nhỏ nhất của 3 hình (cùng 6 kiểu, đổi hàm chi phí) — đối chiếu brute
def perim_six(grid):
    n, m = len(grid), len(grid[0]); rows = grid
    def P(r1, r2, c1, c2):
        pts = [(i, j) for i in range(r1, r2 + 1) for j in range(c1, c2 + 1) if rows[i][j]]
        if not pts:
            return 4                                  # vùng rỗng: vẫn phải đặt một hình 1 × 1 (chu vi 4)
        return 2 * ((max(p[0] for p in pts) - min(p[0] for p in pts) + 1) + (max(p[1] for p in pts) - min(p[1] for p in pts) + 1))
    best = float('inf')
    for i in range(n - 2):
        for j in range(i + 1, n - 1):
            best = min(best, P(0, i, 0, m - 1) + P(i + 1, j, 0, m - 1) + P(j + 1, n - 1, 0, m - 1))
    for i in range(m - 2):
        for j in range(i + 1, m - 1):
            best = min(best, P(0, n - 1, 0, i) + P(0, n - 1, i + 1, j) + P(0, n - 1, j + 1, m - 1))
    for i in range(n - 1):
        for j in range(m - 1):
            best = min(best, P(0, i, 0, m - 1) + P(i + 1, n - 1, 0, j) + P(i + 1, n - 1, j + 1, m - 1),
                       P(i + 1, n - 1, 0, m - 1) + P(0, i, 0, j) + P(0, i, j + 1, m - 1),
                       P(0, n - 1, 0, j) + P(0, i, j + 1, m - 1) + P(i + 1, n - 1, j + 1, m - 1),
                       P(0, n - 1, j + 1, m - 1) + P(0, i, 0, j) + P(i + 1, n - 1, 0, j))
    return best

def perim_brute(grid):
    n, m = len(grid), len(grid[0])
    R = [(r1, r2, c1, c2) for r1 in range(n) for r2 in range(r1, n) for c1 in range(m) for c2 in range(c1, m)]
    C = [frozenset((i, j) for i in range(r[0], r[1] + 1) for j in range(r[2], r[3] + 1)) for r in R]
    ones = [(i, j) for i in range(n) for j in range(m) if grid[i][j]]; best = float('inf')
    for a, b, c in itertools.combinations(range(len(R)), 3):
        if C[a] & C[b] or C[a] & C[c] or C[b] & C[c]:
            continue
        U = C[a] | C[b] | C[c]
        if all(o in U for o in ones):
            best = min(best, sum(2 * (R[x][1] - R[x][0] + 1 + R[x][3] - R[x][2] + 1) for x in (a, b, c)))
    return best

done = 0
for _ in range(120):
    n = rng.randint(1, 3); m = rng.randint(1, 4)
    if n * m < 3:
        continue
    g = [[int(rng.random() < 0.5) for _ in range(m)] for _ in range(n)]
    if sum(map(sum, g)) < 3:
        continue
    assert perim_six(g) == perim_brute(g), g
    done += 1
print('Biến thể chu vi: 6 kiểu khớp brute trên', done, 'lưới.')

Biến thể chu vi: 6 kiểu khớp brute trên 41 lưới.


**Cẩn thận:** với chu vi, quy ước "vùng rỗng = 0" **không** còn an toàn — cắt đôi hình bao `1 × 2` (chu vi 6) thành hai ô đơn tốn `4 + 4 = 8`, nên lời giải "chỉ 2 hình" có thể rẻ hơn mọi lời giải 3 hình thật. Ở đây vùng rỗng phải trả chi phí của hình hợp lệ nhỏ nhất đặt trong vùng (ô `1 × 1`, chu vi 4) — chính xác theo đề. Với diện tích, quy ước 0 an toàn nhờ lập luận ở mục 10 (cũng có thể dùng 1, kết quả không đổi).

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | Brute bộ ba hình | rất lớn | Oracle, không giả thiết |
| 2 | 6 kiểu chia | `O((n²+m²+nm)·n)` | **Nộp bài** |
| 3 | 2 kiểu + xoay | như 2 | Ít code lặp |
| 4 | DP guillotine | `O(n²m²(n+m)k²)` | Tổng quát `k ≤ 4` |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `grid = [[1, 1, 1]]`.

**Bài 2 (tính tay):** `grid = [[1,0,0,1],[0,0,0,0],[1,0,0,1]]`.

**Bài 3 (điền TODO):** trả về luôn **ba hình chữ nhật** `(r1, r2, c1, c2)` của một lời giải tối ưu.

**Bài 4 (tự viết lại):** phiên bản **hai** hình chữ nhật.

**Bài 5 (mini challenge):** phiên bản **bốn** hình chữ nhật bằng DP guillotine; kiểm bằng brute trên lưới nhỏ.

In [18]:
def three_rects_todo(grid):
    # TODO
    raise NotImplementedError

def two_rects_todo(grid):
    # TODO
    raise NotImplementedError

def four_rects_todo(grid):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Ba ô đơn liền nhau. (2) Ba dải dọc độ rộng 1. (3) `3`.

**Bài 2.** (1) Bốn góc, chỉ có 3 hình ⇒ một hình phải chứa 2 góc. (2) Hai góc cùng cột: hình `3 × 1`; cùng hàng: `1 × 4`. (3) `3 + 1 + 1 = 5`.

**Bài 3.** (1) Bên cạnh tổng, lưu ba vùng. (2) Với mỗi vùng tính hình bao thật (toạ độ). (3) Kiểm: không chồng nhau, phủ hết, tổng = đáp án.

**Bài 4.** (1) Một nhát cắt ngang hoặc dọc. (2) `min` của `A(trên) + A(dưới)` và `A(trái) + A(phải)`. (3) Kiểm brute bộ đôi.

**Bài 5.** (1) `f(vùng, 4)` với chia `1+3, 2+2, 3+1`. (2) Bốn hình rời nhau luôn có nhát cắt guillotine. (3) Brute bộ bốn trên lưới `2 × 4`, `3 × 3`.

## 19. Đáp án tham khảo cho bài tập

In [19]:
# Bài 1, 2, 3
assert Solution().minimumSum([[1, 1, 1]]) == 3
assert Solution().minimumSum([[1, 0, 0, 1], [0, 0, 0, 0], [1, 0, 0, 1]]) == 5

def bbox(grid, r1, r2, c1, c2):
    pts = [(i, j) for i in range(r1, r2 + 1) for j in range(c1, c2 + 1) if grid[i][j]]
    if not pts:
        return None
    return (min(p[0] for p in pts), max(p[0] for p in pts), min(p[1] for p in pts), max(p[1] for p in pts))

def three_rects(grid):
    n, m = len(grid), len(grid[0]); parts = []
    for i in range(n - 2):
        for j in range(i + 1, n - 1):
            parts.append([(0, i, 0, m - 1), (i + 1, j, 0, m - 1), (j + 1, n - 1, 0, m - 1)])
    for i in range(m - 2):
        for j in range(i + 1, m - 1):
            parts.append([(0, n - 1, 0, i), (0, n - 1, i + 1, j), (0, n - 1, j + 1, m - 1)])
    for i in range(n - 1):
        for j in range(m - 1):
            parts += [[(0, i, 0, m - 1), (i + 1, n - 1, 0, j), (i + 1, n - 1, j + 1, m - 1)],
                      [(i + 1, n - 1, 0, m - 1), (0, i, 0, j), (0, i, j + 1, m - 1)],
                      [(0, n - 1, 0, j), (0, i, j + 1, m - 1), (i + 1, n - 1, j + 1, m - 1)],
                      [(0, n - 1, j + 1, m - 1), (0, i, 0, j), (i + 1, n - 1, 0, j)]]
    best = None
    for p in parts:
        boxes = [bbox(grid, *reg) for reg in p]
        if None in boxes:
            continue                                   # chỉ nhận kiểu chia mà cả 3 vùng đều có số 1
        s = sum((b[1] - b[0] + 1) * (b[3] - b[2] + 1) for b in boxes)
        if best is None or s < best[0]:
            best = (s, boxes)
    return best[1]

for _ in range(150):
    n = rng.randint(1, 5); m = rng.randint(1, 5)
    g = [[int(rng.random() < 0.4) for _ in range(m)] for _ in range(n)]
    if sum(map(sum, g)) < 3:
        continue
    bx = three_rects(g)
    cells = [{(i, j) for i in range(b[0], b[1] + 1) for j in range(b[2], b[3] + 1)} for b in bx]
    assert not (cells[0] & cells[1] or cells[0] & cells[2] or cells[1] & cells[2])
    assert all(g[i][j] == 0 or any((i, j) in c for c in cells) for i in range(n) for j in range(m))
    assert sum(map(len, cells)) == Solution().minimumSum(g)
print('Bài 1: 3 | Bài 2: 5 | Bài 3: ba hình hợp lệ, tổng tối ưu. Ví dụ 1:', three_rects(examples[0][0]))

Bài 1: 3 | Bài 2: 5 | Bài 3: ba hình hợp lệ, tổng tối ưu. Ví dụ 1: [(0, 1, 0, 0), (1, 1, 1, 1), (0, 1, 2, 2)]


In [20]:
# Bài 4, 5
def two_rects(grid):
    n, m = len(grid), len(grid[0]); A = make_area(grid); best = float('inf')
    for i in range(n - 1):
        best = min(best, A(0, i, 0, m - 1) + A(i + 1, n - 1, 0, m - 1))
    for j in range(m - 1):
        best = min(best, A(0, n - 1, 0, j) + A(0, n - 1, j + 1, m - 1))
    return best

def brute_k(grid, k):
    n, m = len(grid), len(grid[0])
    R = [(r1, r2, c1, c2) for r1 in range(n) for r2 in range(r1, n) for c1 in range(m) for c2 in range(c1, m)]
    C = [frozenset((i, j) for i in range(r[0], r[1] + 1) for j in range(r[2], r[3] + 1)) for r in R]
    ones = frozenset((i, j) for i in range(n) for j in range(m) if grid[i][j]); best = float('inf')
    for combo in itertools.combinations(range(len(R)), k):
        tot = sum(len(C[x]) for x in combo)
        if tot >= best:
            continue
        U = frozenset().union(*(C[x] for x in combo))
        if len(U) == tot and ones <= U:
            best = tot
    return best

for _ in range(80):
    n = rng.randint(1, 3); m = rng.randint(1, 4)
    if n * m < 2:
        continue
    g = [[int(rng.random() < 0.5) for _ in range(m)] for _ in range(n)]
    if sum(map(sum, g)) < 2:
        continue
    assert two_rects(g) == brute_k(g, 2) == guillotine(g, 2)
for g in ([[1, 0, 1, 1], [1, 1, 0, 1]], [[1, 0, 1], [0, 1, 0], [1, 0, 1]], [[1, 1, 0, 1], [0, 1, 1, 1]]):
    assert guillotine(g, 4) == brute_k(g, 4), g
print('Bài 4 (hai hình) khớp brute | Bài 5 (bốn hình, guillotine) khớp brute trên các lưới thử.')

Bài 4 (hai hình) khớp brute | Bài 5 (bốn hình, guillotine) khớp brute trên các lưới thử.


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
A(r1, r2, c1, c2) = hình bao số 1 (0 nếu rỗng)
ba dải ngang:  i < j            : A(0,i) + A(i+1,j) + A(j+1,n-1)   (toàn cột)
ba dải dọc:    i < j            : tương tự theo cột
chữ T (4 kiểu) với (i, j)       : một dải + nửa còn lại chia theo hướng kia
```

**Checklist:**

- [ ] Đủ **6** kiểu (2 kiểu dải + 4 kiểu chữ T)?
- [ ] Vị trí cắt bắt đầu từ 0 (`range(n − 1)`, `range(n − 2)`)?
- [ ] Vùng rỗng trả 0 (an toàn vì có ≥ 3 số 1)?
- [ ] Hình bao tính trong **vùng**, không phải toàn lưới?